In [1]:
import numpy as np

# 1. Define the Helper Functions
def display_pattern(pattern, name):
    """Visualizes a 1D array as a 3x3 grid"""
    print(f"\n--- {name} ---")
    grid = pattern.reshape(3, 3)
    # Replace 1 with '█' and -1 with '.' for visibility
    for row in grid:
        print(" ".join(["█" if x == 1 else "." for x in row]))

def activation_function(x):
    """Bipolar Step Function: Returns 1 if x > 0, else -1"""
    return np.where(x > 0, 1, -1)

# 2. Define the Data (The "Memories")
# Pattern 1: A cross 'X'
X_pattern = np.array([
    1, -1,  1,
   -1,  1, -1,
    1, -1,  1
])
# Target for X: Class A
Y_target_A = np.array([-1, 1]) 

# Pattern 2: A Plus sign '+'
Plus_pattern = np.array([
   -1,  1, -1,
    1,  1,  1,
   -1,  1, -1
])
# Target for Plus: Class B
Y_target_B = np.array([1, -1])

# Store them in lists for the loop
inputs = [X_pattern, Plus_pattern]
targets = [Y_target_A, Y_target_B]

# 3. Apply the Equation: W = sum( x * y.T )
num_inputs = len(X_pattern)  # 9 neurons
num_outputs = len(Y_target_A) # 2 neurons

# Initialize Weight Matrix with zeros
W = np.zeros((num_inputs, num_outputs))

print("Learning patterns...")
for i in range(len(inputs)):
    x_k = inputs[i].reshape(-1, 1) # Make sure it is a column vector (9x1)
    y_k = targets[i].reshape(-1, 1) # Make sure it is a column vector (2x1)
    
    # THE KEY EQUATION: Outer Product
    # np.dot(x, y.T) implements x(y)^T
    weight_update = np.dot(x_k, y_k.T)
    
    # Summation part of the equation
    W += weight_update

print("Training Complete. Weights stored.")

# 4. Testing Phase (Retrieval)
# Let's create a BROKEN 'X' (missing the center pixel)
noisy_input = np.array([
    1, -1,  1,
   -1, -1, -1,  # Center pixel is -1 (off) instead of 1 (on)
    1, -1,  1
])

display_pattern(noisy_input, "Input: Noisy/Broken X")

# 5. Prediction Calculation
# Equation: Output = Activation( Input * W )
raw_output = np.dot(noisy_input, W)
final_output = activation_function(raw_output)

print(f"\nRaw Weighted Sum: {raw_output}")
print(f"Final Prediction: {final_output}")

# Check result
if np.array_equal(final_output, Y_target_A):
    print(">> Result: Correctly identified as Pattern 'X' (Class A)")
elif np.array_equal(final_output, Y_target_B):
    print(">> Result: Identified as Pattern '+' (Class B)")
else:
    print(">> Result: Unknown")

Learning patterns...
Training Complete. Weights stored.

--- Input: Noisy/Broken X ---
█ . █
. . .
█ . █

Raw Weighted Sum: [-16.  16.]
Final Prediction: [-1  1]
>> Result: Correctly identified as Pattern 'X' (Class A)


In [2]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Helper Functions
def compute_energy(w, state):
    """
    Computes the energy of the current state.
    Equation: E = -0.5 * sum(sum(w_ij * x_i * x_j))
    Matrix form: E = -0.5 * x.T * W * x
    """
    return -0.5 * np.dot(state.T, np.dot(w, state))

def train_hopfield(pattern):
    """
    Hebbian Learning Rule: W = x * x.T
    We subtract the identity matrix (np.eye) because neurons don't connect to themselves (w_ii = 0).
    """
    n = len(pattern)
    w = np.outer(pattern, pattern) 
    np.fill_diagonal(w, 0) # Zero out diagonal (no self-connections)
    return w

def update_network(w, state):
    """
    Synchronous update for simplicity in this demo.
    (In strict theory, asynchronous is preferred to prevent oscillation, 
     but synchronous often works fine for simple patterns).
    """
    raw_output = np.dot(w, state)
    new_state = np.where(raw_output >= 0, 1, -1)
    return new_state

# 2. Define a Pattern (4x4 Chessboard)
# +1 = Light, -1 = Dark
pattern_size = 4
target_pattern = np.array([
     1, -1,  1, -1,
    -1,  1, -1,  1,
     1, -1,  1, -1,
    -1,  1, -1,  1
])

# 3. Train the Weights (Hebbian Learning)
W = train_hopfield(target_pattern)

# 4. Create a Noisy Start State
# We flip 4 random bits (significant noise)
noisy_state = target_pattern.copy()
np.random.seed(42) # Fixed seed for reproducibility
noise_indices = np.random.choice(len(target_pattern), 5, replace=False)
noisy_state[noise_indices] *= -1 

print("System initialized.")
print(f"Target Pattern Energy should be lowest.")

# 5. Run the Network and Track Energy
current_state = noisy_state.copy()
history_energy = []

# Calculate initial energy
e = compute_energy(W, current_state)
history_energy.append(e)
print(f"\nStep 0 (Noisy Input): Energy = {e}")

# Iterate until convergence
for i in range(1, 6):
    current_state = update_network(W, current_state)
    e = compute_energy(W, current_state)
    history_energy.append(e)
    
    print(f"Step {i}: Energy = {e}")
    
    # Check if we match the target
    if np.array_equal(current_state, target_pattern):
        print(f"\n>> Converged to Target Pattern at Step {i}!")
        break

# 6. Verify Results
target_energy = compute_energy(W, target_pattern)
print(f"\nTheoretical Minimum Energy (Target): {target_energy}")

# Visualization of states
def print_grid(state, title):
    grid = state.reshape(pattern_size, pattern_size)
    print(f"\n--- {title} ---")
    for row in grid:
        print(" ".join(["#" if x==1 else "." for x in row]))

print_grid(noisy_state, "Noisy Start")
print_grid(current_state, "Recovered State")

System initialized.
Target Pattern Energy should be lowest.

Step 0 (Noisy Input): Energy = -10.0
Step 1: Energy = -120.0

>> Converged to Target Pattern at Step 1!

Theoretical Minimum Energy (Target): -120.0

--- Noisy Start ---
. # # .
. . . #
# . # .
. . # #

--- Recovered State ---
# . # .
. # . #
# . # .
. # . #
